# Eye-movement span — tool 1 (collect blocks → deg/px characteristics)

Map the oculomotor span of many analyzed blocks.

1. **Browse** and multi-select block folders (same UI as the jitter mount pipeline, but eligibility is *eye CSV present*, not a jitter report).
2. **Save** `configs/eye_span_blocks.yaml` (paper `animals:` schema).
3. **Compute** per-eye span characteristics:
   - **full range** = max − min
   - **p95 clip** = p5–p95 (configurable) on each axis, then `hypot`
   - units: **degrees** (Kerr `k_phi`/`k_theta`) and **pixels** (`center_x`/`center_y`)

Tool 2 (µm scaling + jitter ratio) stays in `development/jitter_mount_pipeline.ipynb` §7.

Outputs: `outputs/<run>/metadata/eye_span_characteristics.csv`.

## 0. Setup

In [ ]:
%matplotlib inline
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO = Path.cwd()
if not (REPO / "src" / "eye_tracking_system_tools").is_dir():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "eye_tracking_system_tools").is_dir():
            REPO = p
            break

sys.path.insert(0, str(REPO / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(REPO / ".mplconfig"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(exist_ok=True)

from eye_tracking_system_tools.analysis.block_registry import read_paper_registry
from eye_tracking_system_tools.analysis.jitter_gui import JitterBlockBrowser
from eye_tracking_system_tools.analysis.run_layout import resolve_run_dir
from eye_tracking_system_tools.analysis.span_gui import SpanCharacteristicsPanel

REGISTRY = REPO / "configs" / "eye_span_blocks.yaml"
OUT_ROOT = REPO / "outputs"
TAG = ""  # empty → span_latest (overwrite); e.g. "cohort_v1" → span_cohort_v1

RUN = resolve_run_dir(OUT_ROOT, TAG or None, prefix="span", default_name="span_latest")
print("REPO    :", REPO)
print("registry:", REGISTRY)
print("run     :", RUN.run_dir)
print("  metadata:", RUN.metadata_dir)

## 1. Collect blocks

Reuses the jitter filesystem browser with `require="eye"` (✓ = eye CSV with pupil center and/or Kerr angles) and `registry_format="paper"` so the YAML is an `animals:` map shared with the flexible paper tool.

Mount tags are optional here (kept for convenience / mouse auto-tag); they are not used in the deg/px table.

In [ ]:
browser = JitterBlockBrowser(
    REGISTRY,
    repo=REPO,
    registry_format="paper",
    require="eye",
)
browser

In [ ]:
browser.save()
specs = read_paper_registry(REGISTRY)
print(f"{len(specs)} block(s) in {REGISTRY}")
for s in specs[:12]:
    print(f"  {s.block_key}: {s.block_path}")
if len(specs) > 12:
    print(f"  … +{len(specs) - 12} more")

## 2. Span characteristics (degrees + pixels)

For each eye:

| Column prefix | Meaning |
|---------------|---------|
| `full_*` | max − min on that axis (or hypot of the two axis ranges) |
| `p95_*` | p5–p95 clip (change lo/hi below) |
| `*_deg` | recentered Kerr `k_phi` / `k_theta` |
| `*_px` | pupil `center_x` / `center_y` |

`*_diam_*` is `2 ×` the radial distance from the median position at the same clip (full → max radius; p95 → radius at the hi percentile).

Blocks without Kerr angles still get pixel spans; deg columns stay NaN.

In [ ]:
specs = read_paper_registry(REGISTRY)
panel = SpanCharacteristicsPanel(specs, RUN.metadata_dir, lo=5.0, hi=95.0)
panel
# Click **Compute spans**, or run: panel.compute()

In [ ]:
table = panel.compute()
print("wrote:", panel.out_path)
table.head()

## Notes

| Piece | Where |
|-------|-------|
| Registry | `configs/eye_span_blocks.yaml` |
| Table | `outputs/<run>/metadata/eye_span_characteristics.csv` |
| Browser | `analysis/jitter_gui.JitterBlockBrowser(require="eye")` |
| Metrics | `analysis/eye_movement_span.collect_span_characteristics` |
| Widget | `analysis/span_gui.SpanCharacteristicsPanel` |

Headless::

```bash
PYTHONPATH=src python -m eye_tracking_system_tools.analysis.eye_movement_span \
  --registry configs/eye_span_blocks.yaml --out-root outputs --tag ""
```

You can also point `--registry` at an existing paper registry (`configs/paper_blocks.yaml`) or the jitter mount YAML.